# MobileNetV3-Large: Instructor Test Submission

Notebook นี้ใช้สำหรับชุด Test จริงของอาจารย์เท่านั้น ไม่มีการฝึก ไม่มีการใช้เฉลย และไม่คำนวณ metrics

ขั้นตอน: Mount Drive → ตั้ง path และชื่อคอลัมน์กลุ่ม → โหลด checkpoint → เลือก Test ZIP → Run all → รับ `submission.csv` ตาม template ของอาจารย์

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi


## 1. ตั้งค่า

แก้ `TEST_ZIP_PATH` หากอัปโหลด Test ZIP ไว้ใน Drive หาก path ไม่พบ Notebook จะเปิดหน้าต่างให้อัปโหลดจากเครื่องอัตโนมัติ

In [ ]:
from pathlib import Path

MODEL_PATH = Path('/content/drive/MyDrive/dl/thai_char_person3_outputs/mobilenetv3_large_best.pt')
LABELS_PATH = Path('/content/drive/MyDrive/dl/person3_evaluation/labels.csv')
DATA_DICT_PATH = Path('/content/drive/MyDrive/dl/Project_1-data_dict.csv')
SUBMISSION_TEMPLATE_PATH = Path('/content/drive/MyDrive/dl/Project_1-InputOutput.csv')

# เปลี่ยนให้ตรงกับชื่อคอลัมน์กลุ่มของตัวเองใน Project_1-InputOutput.csv
TEAM_COLUMN = 'test'

# เปลี่ยนชื่อนี้ให้ตรงกับ ZIP ของอาจารย์ หรือปล่อยไว้เพื่อเลือกอัปโหลดจากเครื่อง
TEST_ZIP_PATH = Path('/content/drive/MyDrive/dl/INSTRUCTOR_TEST.zip')
TEST_FOLDER_PATH = Path('/content/drive/MyDrive/dl/INSTRUCTOR_TEST')

OUTPUT_DIR = Path('/content/drive/MyDrive/dl/submission_outputs/person3_mobilenetv3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True

assert MODEL_PATH.exists(), f'ไม่พบ checkpoint: {MODEL_PATH}'
assert DATA_DICT_PATH.exists(), f'ไม่พบ data dictionary: {DATA_DICT_PATH}'
assert SUBMISSION_TEMPLATE_PATH.exists(), f'ไม่พบ submission template: {SUBMISSION_TEMPLATE_PATH}'
print('Model:', MODEL_PATH)
print('Data dictionary:', DATA_DICT_PATH)
print('Submission template:', SUBMISSION_TEMPLATE_PATH)
print('Team column:', TEAM_COLUMN)
print('Output:', OUTPUT_DIR)


## 2. เตรียม Test Images

In [ ]:
import shutil
import re
import time

import pandas as pd
import torch
import torch.nn as nn
from google.colab import files
from PIL import Image, ImageFile, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')

LOCAL_TEST_DIR = Path('/content/instructor_test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

if not TEST_ZIP_PATH.exists() and not TEST_FOLDER_PATH.exists():
    print('ไม่พบชุด Test ใน Drive กรุณาเลือก ZIP ที่อาจารย์ให้')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('ยังไม่ได้อัปโหลด Test ZIP')
    uploaded_name = next(iter(uploaded))
    TEST_ZIP_PATH = Path('/content') / uploaded_name
    if TEST_ZIP_PATH.suffix.lower() != '.zip':
        raise ValueError(f'กรุณาอัปโหลดไฟล์ .zip: {TEST_ZIP_PATH.name}')

if TEST_ZIP_PATH.exists():
    if LOCAL_TEST_DIR.exists():
        shutil.rmtree(LOCAL_TEST_DIR)
    print('กำลังแตก Test ZIP...')
    shutil.unpack_archive(TEST_ZIP_PATH, LOCAL_TEST_DIR)
    TEST_ROOT = LOCAL_TEST_DIR
else:
    TEST_ROOT = TEST_FOLDER_PATH

image_files = sorted(
    path for path in TEST_ROOT.rglob('*')
    if (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
        and '__MACOSX' not in path.parts
        and not path.name.startswith('._')
    )
)
if not image_files:
    raise RuntimeError(f'ไม่พบภาพภายใน {TEST_ROOT}')
print('Test root:', TEST_ROOT)
print(f'Images: {len(image_files):,}')


## 3. โหลด Model และ Preprocessing

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
required_keys = {'model_state_dict', 'class_to_idx', 'image_size'}
missing_keys = required_keys.difference(checkpoint)
if missing_keys:
    raise KeyError(f'Checkpoint ขาด keys: {sorted(missing_keys)}')

class_to_idx = {str(code): int(index) for code, index in checkpoint['class_to_idx'].items()}
num_classes = len(class_to_idx)
idx_to_code = [None] * num_classes
for code, index in class_to_idx.items():
    idx_to_code[index] = code
if num_classes != 80 or any(code is None for code in idx_to_code):
    raise ValueError('Checkpoint class mapping ไม่ถูกต้อง')

image_size = int(checkpoint['image_size'])
model = models.mobilenet_v3_large(weights=None)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device).eval()

class Letterbox:
    def __init__(self, size, fill=(255, 255, 255)):
        self.size = (size, size)
        self.fill = fill

    def __call__(self, image):
        image = image.convert('RGB')
        return ImageOps.pad(
            image, self.size, method=Image.Resampling.BILINEAR,
            color=self.fill, centering=(0.5, 0.5),
        )

eval_transform = transforms.Compose([
    Letterbox(image_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

labels_df = pd.read_csv(LABELS_PATH) if LABELS_PATH.exists() else None
if labels_df is not None:
    labels_df['tis620'] = labels_df['tis620'].astype(str)
    char_by_code = dict(zip(labels_df.tis620, labels_df.char))
    name_by_code = dict(zip(labels_df.tis620, labels_df.name_en))
else:
    char_by_code = {}
    name_by_code = {}

def read_project_csv(path):
    for encoding in ('utf-8-sig', 'cp874', 'tis-620'):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise UnicodeDecodeError('unknown', b'', 0, 1, f'อ่านไฟล์ไม่สำเร็จ: {path}')

data_dict_df = read_project_csv(DATA_DICT_PATH).dropna(subset=['folder', 'class_id']).copy()
data_dict_df['folder'] = data_dict_df['folder'].astype(int).astype(str)
data_dict_df['class_id'] = data_dict_df['class_id'].astype(int)
folder_to_project_class_id = dict(zip(data_dict_df.folder, data_dict_df.class_id))

allowed_indices = [
    class_to_idx[folder]
    for folder in folder_to_project_class_id
    if folder in class_to_idx
]
if len(folder_to_project_class_id) != 72 or len(allowed_indices) != 72:
    raise ValueError(
        f'ต้องพบ 72 class_id ใน data dictionary และ checkpoint: '
        f'{len(folder_to_project_class_id)} / {len(allowed_indices)}'
    )
allowed_indices_tensor = torch.tensor(allowed_indices, device=device)

print('Model: MobileNetV3-Large')
print('Classes:', num_classes)
print('Input size:', image_size)
if 'best_val_acc' in checkpoint:
    print(f"Checkpoint validation accuracy: {float(checkpoint['best_val_acc']):.2%}")


## 4. Predict และสร้าง Submission

ไฟล์หลัก `submission.csv` คงรูปแบบของ `Project_1-InputOutput.csv` และกรอกเฉพาะคอลัมน์กลุ่มของเราเป็น `class_id` 3 หลัก เช่น `101`, `205`, `301` ส่วน `submission_detailed.csv` เก็บรายละเอียดเพิ่มเติมสำหรับตรวจสอบ

In [ ]:
import re

class TestImages(Dataset):
    def __init__(self, root, paths, transform):
        self.root = Path(root)
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as image:
            tensor = self.transform(image)
        return tensor, path.relative_to(self.root).as_posix()

test_dataset = TestImages(TEST_ROOT, image_files, eval_transform)
test_loader = DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=2,
    pin_memory=device.type == 'cuda', persistent_workers=True,
)

rows = []
partial_details_path = OUTPUT_DIR / 'submission_detailed_partial.csv'
started = time.perf_counter()
with torch.inference_mode():
    for inputs, relative_paths in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'
        ):
            logits = model(inputs)
            allowed_probabilities = logits.index_select(1, allowed_indices_tensor).softmax(dim=1).cpu()
        allowed_positions = allowed_probabilities.argmax(dim=1)
        indices = allowed_indices_tensor[allowed_positions.to(device)].cpu().numpy()
        confidences = allowed_probabilities.max(dim=1).values.numpy()
        for relative_path, index, confidence in zip(relative_paths, indices, confidences):
            code = idx_to_code[int(index)]
            rows.append({
                'filename': Path(relative_path).name,
                'relative_path': relative_path,
                'model_index': int(index),
                'tis620': code,
                'project_class_id': folder_to_project_class_id[code],
                'char': char_by_code.get(code, ''),
                'name_en': name_by_code.get(code, ''),
                'confidence': float(confidence),
            })
        # Save after every prediction batch so Colab interruption does not lose completed work.
        pd.DataFrame(rows).to_csv(partial_details_path, index=False, encoding='utf-8-sig')
elapsed = time.perf_counter() - started
details_df = pd.DataFrame(rows)

template_df = read_project_csv(SUBMISSION_TEMPLATE_PATH)
if TEAM_COLUMN not in template_df.columns:
    raise ValueError(
        f'ไม่พบ TEAM_COLUMN={TEAM_COLUMN!r} ใน template. '
        f'เลือกหนึ่งค่าใน: {template_df.columns.tolist()[3:]}'
    )

if len(template_df) != len(details_df):
    raise ValueError(
        f'จำนวนภาพ Test ({len(details_df):,}) ไม่เท่ากับจำนวนแถวใน template ({len(template_df):,}). '
        'ตรวจว่าเลือก Test ZIP และ Project_1-InputOutput.csv ชุดจริงของอาจารย์แล้ว'
    )

def file_stem(value):
    return Path(str(value).replace('\\', '/')).stem.casefold()

def trailing_number(value):
    matches = re.findall(r'\d+', file_stem(value))
    return int(matches[-1]) if matches else None

details_df['file_stem'] = details_df.filename.map(file_stem)
if details_df.file_stem.duplicated().any():
    raise ValueError('ชื่อไฟล์ Test ซ้ำกันหลังตัดนามสกุล จึงจับคู่กับ gt_path ไม่ได้')
prediction_by_stem = dict(zip(details_df.file_stem, details_df.project_class_id))
template_stems = template_df['gt_path'].map(file_stem)
project_predictions = template_stems.map(prediction_by_stem)

# Some instructor ZIPs use a different filename prefix. Fall back to a unique trailing image number.
if project_predictions.isna().any():
    details_df['file_number'] = details_df.filename.map(trailing_number)
    if details_df.file_number.isna().any() or details_df.file_number.duplicated().any():
        raise ValueError('ชื่อภาพ Test ต้องมีเลขท้ายที่ไม่ซ้ำกัน เพื่อใช้จับคู่กับ template')
    prediction_by_number = dict(zip(details_df.file_number, details_df.project_class_id))
    template_numbers = template_df['gt_path'].map(trailing_number)
    project_predictions = project_predictions.fillna(template_numbers.map(prediction_by_number))
if project_predictions.isna().any():
    missing = template_df.loc[project_predictions.isna(), 'gt_path'].head(10).tolist()
    raise ValueError(
        f'จับคู่ภาพ Test กับ template ไม่ครบ: {project_predictions.isna().sum()} แถว. '
        f'ตัวอย่าง: {missing}'
    )

submission_df = template_df.copy()
submission_df[TEAM_COLUMN] = project_predictions.astype(int)
submission_path = OUTPUT_DIR / 'submission.csv'
details_path = OUTPUT_DIR / 'submission_detailed.csv'
submission_df.to_csv(submission_path, index=False, encoding='utf-8-sig')
details_df.to_csv(details_path, index=False, encoding='utf-8-sig')

print(f'Predicted: {len(details_df):,} images')
print(f'Filled {TEAM_COLUMN!r}: {len(submission_df):,} rows')
print(f'Inference time: {elapsed:.2f} seconds')
print('Saved:', submission_path)
print('Progress checkpoint:', partial_details_path)
display(submission_df.head(10))

if AUTO_DOWNLOAD:
    files.download(str(submission_path))


## ก่อนส่ง

1. อัปโหลด `Project_1-data_dict.csv` และ `Project_1-InputOutput.csv` ไปยัง path ที่ระบุในหัวข้อ 1
2. แก้ `TEAM_COLUMN` ให้ตรงกับชื่อคอลัมน์กลุ่มของตัวเองแบบตัวอักษรต่ออักษร
3. ส่งเฉพาะค่าจากคอลัมน์ `TEAM_COLUMN` ไปกรอกใน Google Sheet ของอาจารย์ หรือใช้ `submission.csv` เป็นไฟล์อ้างอิง

ห้ามกรอก TIS-620 หรือ model index; ค่าที่ส่งต้องเป็น `class_id` 3 หลักตาม Project 1 เท่านั้น